# dispatch-back-fn-from-recipe composite — cx21: dispatch back_fn over each (argnum, parent) in parents dict

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `dispatch-back-fn-from-recipe`, `parents-dict-by-argidx`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "dispatch-back-fn-from-recipe"
DD_ATOM_IDS = ["dispatch-back-fn-from-recipe", "parents-dict-by-argidx"]
DD_SUBTOPICS = ["Backprop: dispatch back fn from recipe", "Backprop: Parents dict by argidx"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing recipe dispatch with the parents-dict-by-argidx walk

When the reverse pass needs the gradients for a non-leaf node's inputs:

- **`parents-dict-by-argidx`** — `recipe.parents` is the
  `{argnum: parent_tensor}` dict (non-Tensor args already filtered,   ORIGINAL argnum preserved as the key).
- **`dispatch-back-fn-from-recipe`** — for each `(argnum, parent)`   pair, look up `back_funcs[(recipe.func, argnum)]`.

```python
def dispatch_all(node, back_funcs):
    triples = []
    for argnum, parent in node.recipe.parents.items():   # by-argidx walk
        back_fn = back_funcs[(node.recipe.func, argnum)] # argnum-keyed dispatch
        triples.append((argnum, parent, back_fn))
    return triples
```

**Both atoms are joined at the argnum.** The parents-dict's KEY is the
argnum that the dispatcher's LOOKUP needs. If the parents-dict
renumbered (e.g. collapsed `{0: t, 2: u}` to `{0: t, 1: u}`), the
dispatcher would look up `(fn, 1)` for the second parent and get the
WRONG back_fn — that's why parents-dict-by-argidx is strict about
preserving the original index.

**Asymmetric ops are the test.** `multiply` registers `(mul, 0)` and
`(mul, 1)` with the SAME body. `divide` registers `(div, 0)` and `(div,
1)` with DIFFERENT bodies (`grad/y` vs `-grad*x/y**2`). If parents are
indexed wrong, divide gets the wrong derivative.

### Composite Exercise — dispatch back_fn over each (argnum, parent) in parents dict

**Atoms exercised together**: `dispatch-back-fn-from-recipe`, `parents-dict-by-argidx`

Implement `cx21_dispatch_all(args, raw_func, back_funcs)` — given the RAW positional args at forward-call time (a mix of MiniTensors and non-Tensors), the forward function, and the back_funcs registry, return a `list[(argnum, parent, back_fn)]` ready for the back_fn call site. Two atoms compose:

**Step 1 — `parents-dict-by-argidx`.** Build a parents dict using `{idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}`. MUST keep the ORIGINAL argidx — DO NOT renumber.

**Step 2 — `dispatch-back-fn-from-recipe`.** For each `(argnum, parent)` in that dict, look up `back_funcs[(raw_func, argnum)]` and append `(argnum, parent, back_fn)` to the result list.

**Test surface.**
- `multiply(t, 3.0)` → parents `{0: t}`, dispatched to `(mul, 0)`.
- `multiply(3.0, t)` → parents `{1: t}` (argnum stays 1!), dispatched to `(mul, 1)` NOT `(mul, 0)`.
- `divide(x, y)` (asymmetric) → both `(div, 0)` and `(div, 1)` get the right body.
- Missing `(raw_func, argnum)` → propagate `KeyError`.

**A `MiniTensor` class is provided in the test.**

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array; self.requires_grad = requires_grad

def cx21_dispatch_all(args, raw_func, back_funcs) -> list:
    """Build parents dict by argidx, dispatch back_fn for each parent."""
    raise NotImplementedError()

def _test_cx21():
    def log_back(grad_out, out, x): return grad_out / x
    def mul_back0(grad_out, out, x, y): return grad_out * y
    def mul_back1(grad_out, out, x, y): return grad_out * x
    def div_back0(grad_out, out, x, y): return grad_out / y
    def div_back1(grad_out, out, x, y): return -grad_out * x / (y * y)

    BF = {
        (t.log, 0): log_back,
        (t.multiply, 0): mul_back0, (t.multiply, 1): mul_back1,
        (t.multiply, 2): mul_back0, (t.multiply, 3): mul_back1,
        (t.divide, 0): div_back0, (t.divide, 1): div_back1,
    }

    # === single parent (log) ===
    b = MiniTensor(t.tensor([2.0]))
    triples = cx21_dispatch_all((b,), t.log, BF)
    assert len(triples) == 1
    argnum, parent, back_fn = triples[0]
    assert argnum == 0 and parent is b and back_fn is log_back

    # === two parents (multiply) — both back_fns from symmetric op ===
    x = MiniTensor(t.tensor([2.0])); y = MiniTensor(t.tensor([3.0]))
    triples = cx21_dispatch_all((x, y), t.multiply, BF)
    assert len(triples) == 2
    by_argnum = {a: (p, f) for a, p, f in triples}
    assert by_argnum[0] == (x, mul_back0)
    assert by_argnum[1] == (y, mul_back1)

    # === non-Tensor at arg-0 — argnum MUST STAY 1, not collapse to 0 ===
    a = MiniTensor(t.tensor([7.0]))
    triples = cx21_dispatch_all((3.0, a), t.multiply, BF)
    assert len(triples) == 1
    argnum, parent, back_fn = triples[0]
    assert argnum == 1, f'argnum must remain 1 (not collapse to 0); got {argnum}'
    assert parent is a
    assert back_fn is mul_back1, 'must dispatch to mul_back1 (NOT mul_back0)'

    # === asymmetric divide — both back_fns must differ ===
    p, q = MiniTensor(t.tensor([6.0])), MiniTensor(t.tensor([2.0]))
    triples = cx21_dispatch_all((p, q), t.divide, BF)
    by_argnum = {a: (par, f) for a, par, f in triples}
    assert by_argnum[0] == (p, div_back0)
    assert by_argnum[1] == (q, div_back1)
    assert by_argnum[0][1] is not by_argnum[1][1], 'div_back0 != div_back1'

    # === confirm asymmetric math by running the back_fns ===
    grad_p = by_argnum[0][1](t.ones(1), p.array / q.array, p.array, q.array)
    grad_q = by_argnum[1][1](t.ones(1), p.array / q.array, p.array, q.array)
    assert t.allclose(grad_p, t.tensor([0.5])), f'd(p/q)/dp=1/q=0.5; got {grad_p}'
    assert t.allclose(grad_q, t.tensor([-1.5])), f'd(p/q)/dq=-p/q^2=-1.5; got {grad_q}'

    # === non-contiguous parents (5, t, (1,2), u) ===
    u = MiniTensor(t.tensor([1.0]))
    triples = cx21_dispatch_all((5, x, (1, 2), u), t.multiply, BF)
    assert len(triples) == 2
    argnums = sorted(tr[0] for tr in triples)
    assert argnums == [1, 3], f'expected argnums [1, 3]; got {argnums}'

    # === missing registration → KeyError ===
    raised = False
    try: cx21_dispatch_all((x,), t.sin, BF)
    except KeyError: raised = True
    assert raised, 'unregistered (fn, argnum) must propagate KeyError'

    # === all non-Tensors → empty list ===
    triples = cx21_dispatch_all((1.0, 2.0, 'x'), t.multiply, BF)
    assert triples == [], f'all non-Tensors -> empty list, got {triples}'
    _dd_passed.add('cx21')

_test_cx21()

<details><summary>Show solution — cx21</summary>

```python
def cx21_dispatch_all(args, raw_func, back_funcs):
    # Step 1: parents-dict-by-argidx — preserve ORIGINAL idx, skip non-Tensors.
    parents = {
        idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)
    }
    # Step 2: dispatch-back-fn-from-recipe — (raw_func, argnum) key per parent.
    results = []
    for argnum, parent in parents.items():
        back_fn = back_funcs[(raw_func, argnum)]
        results.append((argnum, parent, back_fn))
    return results
```

**`enumerate` BEFORE `isinstance` filter — load-bearing.** Doing it the other way (filter first, then enumerate) re-numbers the survivors. `(3.0, t)` would yield `{0: t}` instead of `{1: t}` — and the dispatcher would look up `(mul, 0)` instead of `(mul, 1)`. For symmetric ops you'd never notice; for divide you'd get the WRONG derivative.

**The argnum threads through three layers.** It's the KEY in `parents-dict-by-argidx`, the LOOKUP component in `dispatch-back-fn-from-recipe`, and the SELECTOR that picks the right back_fn body. Renumber it anywhere and the chain breaks.

**KeyError propagates intentionally.** The caller (the reverse-pass driver) is the right place to decorate the error with context ('node at depth N in graph X'). The dispatcher just signals 'no back_fn registered for this (fn, argnum)'.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx21'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx21',
        'subtopics': ["Backprop: dispatch back fn from recipe", "Backprop: Parents dict by argidx"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()